# 04. Spatial tiling: контекст, blend и границы

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Выведем условия корректной сшивки, исполним оригинальные blend/tiled methods и отделим отсутствие швов от эквивалентности full inference.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT предпосылка:** закончите Lab 3 Part 4 и получите full reconstruction. Spatial tiling отсутствует среди Q4.1–4.7 и является промышленным расширением. Сравниваем с тем же full baseline. [Переход к практике](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='04_spatial_tiling_and_blending';metrics={}
print('Pinned source:',source_hash())


## 1. Зачем overlap и как устроен blend

Tile size Q, stride S, overlap O=Q−S. В latent coordinates encoder требует Q_z=Q/8, S_z=S/8. Смешиваем два предсказания одного глобального pixel/latent position:

$$
y_j=(1-\alpha_j)a_j+\alpha_j b_j,\qquad \alpha_j=j/O,\quad j=0,\ldots,O-1.
$$

Веса суммируются к единице. При a=b результат не меняется. Последняя α=(O−1)/O, а не 1; следующий несмешанный pixel уже берётся из нового tile. Source `blend_*` меняет b **in-place**. Vertical blending выполняется до horizontal; четыре tile contributions в углу зависят от этого порядка.

Обобщённая weighted overlap-add (WOLA) делит сумму weighted predictions на сумму weights. Это другая схема; оригинальные methods ниже не заменяем WOLA молча:

$$
\hat y(p)=\frac{\sum_i w_i(p)y_i(p)}{\sum_iw_i(p)},\qquad \sum_iw_i(p)>0.
$$


In [ ]:
obj=oracle_codec();show_source('blend_h');show_source('blend_v')
a=torch.full((1,1,1,1,8),2.);b=torch.full_like(a,6.)
y=obj.blend_h(a,b.clone(),4).flatten()
expected=torch.tensor([2.,3.,4.,5.,6.,6.,6.,6.]);assert torch.equal(y,expected)
metrics['native_blend_values']=y.tolist()
fig,ax=plt.subplots(figsize=(6,3));ax.plot(range(8),y,'o-');ax.set(xlabel='Tile b local index',ylabel='Blended value',title='Native linear overlap: 2 → 6')
fig.tight_layout();fig.savefig(figure_path(LAB,'blend'));plt.show()


## 2. Exact coordinate oracle: сначала доказываем индексы

Возьмём encoder, который выбирает каждый четвёртый кадр и каждый восьмой spatial pixel, decoder с first-frame preserving repetition. Он не учился и не имеет context. Если tile origins кратны compression ratio и loop полностью покрывает кадр, tiled и full outputs должны совпасть.

Это сильный **index test**, но слабый quality test. Он не выявляет GroupNorm, attention или boundary padding artifacts. Original methods получают оригинальные global coordinate values, поэтому off-by-one станет виден без нейросети.


In [ ]:
x=torch.arange(1*1*17*48*48,dtype=torch.float64).reshape(1,1,17,48,48)
obj=oracle_codec()
full=obj.quant_conv(obj.encoder(x));tiled=obj.tiled_encode(x)
assert torch.equal(full,tiled)
full_dec=obj.decoder(full);tile_dec=obj.tiled_decode(full).sample
assert torch.equal(full_dec,tile_dec)
metrics['aligned_oracle_encode_max_error']=float((full-tiled).abs().max())
metrics['aligned_oracle_decode_max_error']=float((full_dec-tile_dec).abs().max())
show_source('tiled_encode');show_source('tiled_decode')


## 3. Покрытие правого/нижнего края — отдельное условие

Native loop: starts=range(0,L−Q+1,S). Он генерирует только полные tiles. При L≥Q последний покрытый index exclusive:

$$
L_{covered}=Q+S\left\lfloor\frac{L-Q}{S}\right\rfloor.
$$

Missing tail равен (L−Q) mod S. Финальный slice `:latent_width` может отрезать лишнее, но не дорисует отсутствующее. При L<Q список tiles пуст. Сработает ли проблема в приложении, зависит от planner/caller dimensions; проверяем обе части, а не объявляем весь Hunyuan upstream ошибочным.

Ниже L=88, Q=32, S=24: 88 кратно 8, но coverage=80. Пример показывает ограничение **прикреплённой адаптации**, не утверждение статьи о невозможности arbitrary sizes.


In [ ]:
obj=oracle_codec();obj.apply_tiling((1,17,32,32),(8,24,24))
x=torch.zeros(1,1,17,88,88)
got=obj.tiled_encode(x)
coverage=axis_coverage(88,32,24)
assert coverage['missing']==8 and got.shape[-1]==10 # expected 88/8 = 11
metrics['tail_counterexample']={'coverage':coverage,'expected_latent_width':11,'actual':got.shape[-1]}
print(metrics['tail_counterexample'])
# Explicit alternative, not a silent patch of pinned code:
starts=anchored_starts(88,32,24,align=8)
assert starts==[0,24,48,56]
print('Anchored grid:',starts,'variable final overlap:',32-(starts[-1]-starts[-2]))


## 3b. Точный halo+crop на локальной свёртке

Пусть y[i]=Σw[k]x[i+k−r], radius r=1. Для output interval [a,b) достаточно input [a−r,b+r). Добавим halo, применим тот же convolution и вырежем только valid interior. На внешней границе используем тот же zero padding, что и full conv. Тогда каждый выход получает ровно те же summands — это конструктивное доказательство exactness для локального linear operator.

Сравним без halo и с halo. Этот пример намеренно 1D, чтобы индексы можно было проверить вручную; spatial 2D получается декартовым произведением двух interval halos. Добавление GroupNorm нарушит предпосылку локальности.


In [ ]:
x=torch.randn(1,1,32);weight=torch.tensor([[[.2,.6,.2]]])
full=F.conv1d(x,weight,padding=1)
naive=torch.cat([F.conv1d(x[:,:,a:a+8],weight,padding=1) for a in range(0,32,8)],dim=2)
padded=F.pad(x,(1,1));parts=[]
for a in range(0,32,8):
    b=a+8
    parts.append(F.conv1d(padded[:,:,a:b+2],weight,padding=0))
with_halo=torch.cat(parts,dim=2)
assert torch.equal(full,with_halo) and not torch.equal(full,naive)
metrics['halo_local_conv']={'with_halo_max_error':float((full-with_halo).abs().max()),
                           'without_halo_max_error':float((full-naive).abs().max())}
print(metrics['halo_local_conv'])


## 4. Почему overlap не гарантирует full equivalence

Для локального оператора с radius r пространственный halo r и cropping interiors могут дать exact full result (кроме внешних boundaries). Без halo у каждого tile искусственная граница. Blend сглаживает расхождения, но не восстанавливает отсутствующий контекст.

GroupNorm использует статистики всего tile. Mid attention видит только local tile tokens. Даже очень большой конечный convolutional halo не восстанавливает global operation. В статье §4.1.2 дополнительный finetuning с включением/отключением tiling как раз уменьшает этот train/inference mismatch; не делает произвольный checkpoint математически exact.

Проверим ненулевую разницу на random native encoder без attention, где остаётся GroupNorm. Фиксируем одни веса и input: меняется только spatial policy.


In [ ]:
codec=TinyCodec(seed=4);x=synthetic_video(9,48,48)
with torch.inference_mode():
    full=codec._encode(x)
    codec.set_tiles(spatial=True,temporal=False)
    tiled=codec._encode(x)
stat=errors(full,tiled);assert stat['mse']>0
metrics['native_tiny_spatial_difference']=stat
emap=(full-tiled).square().mean((0,1,2)).numpy()
fig,ax=plt.subplots(figsize=(4,3));im=ax.imshow(emap);fig.colorbar(im,ax=ax,label='Latent squared error');ax.set(title='Same random encoder: full vs spatial',xlabel='Latent W',ylabel='Latent H')
fig.tight_layout();fig.savefig(figure_path(LAB,'spatial_error'));plt.show()
print(stat)


## Упражнения и критерий перехода

1. Реализуйте WOLA и докажите, что constant field сохраняется.
2. Сделайте halo+crop для одной обычной spatial convolution и получите exact interior equivalence; затем добавьте GroupNorm.
3. Обработайте L<Q и нечётные H/W через явный pad/crop policy. Переход: oracle exactness, coverage и perceptual quality должны быть тремя отдельными проверками.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
